# makemore 1: bigram counts

Karpathy, *Zero to Hero* #2, 0:00 to ~1:02 (the counting model; the neural-net half is makemore 2, Thursday).

Goal: (1) turn 32,000 names into a table of counts, (2) sample new names from it, (3) score the model with one number, and **know why that number is the right one**.

**Rules:** predict before you run, in the comment provided. Type the code yourself; the `___` blanks are yours.

## Block 0: micrograd README (20 min, from memory)
Carried over from last night. Open `micrograd/README.md`. **Without looking at the log, your notes or Saturday's draft**, add under *Backprop in my own words*:

1. **Why tanh?** (3–4 sentences.) Use your numbers: the no-tanh floor you calculated (28/27) and hit, and the tanh net's 0.0067.
2. **How I know my engine is right.** (3–4 sentences.) The PyTorch cross-check and its number (4.4e-16); why a falling loss isn't evidence (the planted bug); what a cross-check *can't* prove.
3. Tick the PyTorch box.

Paste it in the chat for review **before** you push.

## Block 1: the data (video 0:00 to ~0:15, 15 min)

In [ ]:
import os, urllib.request
import torch
import matplotlib.pyplot as plt
%matplotlib inline

path = '../data/names.txt'
if not os.path.exists(path):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/karpathy/makemore/master/names.txt', path)

words = open(path, 'r').read().splitlines()
print(len(words), words[:5])
print('shortest', min(len(w) for w in words), 'longest', max(len(w) for w in words))

1. A bigram model predicts each character from **only the one before it**. Karpathy wraps each name in a start/end token. **Predict:** how many bigrams does a name of length *n* give? So how many does `emma` give, and which are they?
2. **Predict** the total number of bigrams across all 32,033 names, roughly. (You know the count and you can guess the average length.) Then run.
3. Why does the model need to know where a name **ends**? What would sampling do without it?

In [ ]:
# predict: a name of length n gives ___ bigrams; total across the dataset ≈ ___
total = 0
for w in words[:3]:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        print(ch1, ch2)

for w in words:
    ___   # count every bigram in the dataset
print('total bigrams', total)

## Block 2: the count table (to ~0:30, 15 min)
Map 26 letters plus `.` to the integers 0–26, then fill a 27×27 tensor `N` where `N[i, j]` = how often character j follows character i.

Karpathy first uses two tokens (`<S>`, `<E>`), then switches to one `.`. **Before you watch why**, say what's wasted with two (think about one row and one column of the 28×28 table).

In [ ]:
chars = sorted(list(set(''.join(words))))
stoi = {s: i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}

N = torch.zeros((27, 27), dtype=torch.int32)
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ___   # one line: add this bigram to N

# predict: the most common bigram is ___ ; N[0, 0] = ___ because ___

In [ ]:
plt.figure(figsize=(16, 16))
plt.imshow(N, cmap='Blues')
for i in range(27):
    for j in range(27):
        plt.text(j, i, itos[i] + itos[j], ha='center', va='bottom', color='gray')
        plt.text(j, i, N[i, j].item(), ha='center', va='top', color='gray')
plt.axis('off');

4. Find the five most common bigrams with code (hint: `N.flatten()`, `torch.topk`). Did you predict the top one?
5. **Predict, then check:** `N[0].sum()` (the `.` row) and `N[:, 0].sum()` (the `.` column). Why *must* they be equal, and equal to what?
6. How many cells of `N` are exactly 0? Name one. Hold onto that: it matters in Block 4.

In [ ]:
# predict: N[0].sum() = ___ , N[:, 0].sum() = ___ , zero cells ≈ ___
___

## Break (5 min)

## Block 3: sampling (to ~0:50, 15 min)
Turn each row of counts into a probability distribution, then generate names one character at a time.

1. `N.sum(1, keepdims=True)` has shape (27, 1). **Predict** the shape of `N.sum(1)` without `keepdims`. Then: if you divide `P = N / N.sum(1)` (no keepdims), broadcasting lines the shapes up from the **right**. Which way does the division go wrong: are you normalising rows or columns? Predict before you run the check cell.
2. Why add 1 to every count (`N + 1`)? Leave the answer for Block 4; for now, just do it.

In [ ]:
P = (N + 1).float()
P /= P.sum(1, keepdims=True)

# predict: without keepdims, row 0 of the wrong P sums to ___ (1, or not 1?)
Pwrong = (N + 1).float() / (N + 1).float().sum(1)
print('right: row 0 sums to', P[0].sum().item())
print('wrong: row 0 sums to', Pwrong[0].sum().item(), '| column 0 sums to', Pwrong[:, 0].sum().item())

In [ ]:
g = torch.Generator().manual_seed(2147483647)

for _ in range(10):
    out = []
    ix = 0                      # start at '.'
    while True:
        p = P[ix]
        ix = ___                # one sample from p: torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        if ix == 0:
            break
        out.append(itos[ix])
    print(''.join(out))

3. Now replace `p = P[ix]` with a **uniform** distribution (`torch.ones(27) / 27`) and sample again. Compare the two sets of names. What has the bigram model learnt, and what can it never learn, however much data you give it?

In [ ]:
# predict: uniform names will look like ___ ; and their average length will be about ___ characters (why?)
___

## Block 4: one number for how good the model is (to ~1:02, 20 min)
**This is the block that matters most.** If you're short of time, cut the uniform-sampling cell above, not this.

The model gives a probability to every bigram in the training set. A good model gives the real data **high** probability.

1. **Likelihood** = the product of the probabilities of every bigram in the dataset. Why is the raw product useless to compute with on 228,146 bigrams? (micrograd 1 territory: what does a float do?)
2. Take logs. log is **monotonic**: what does that guarantee about the model that maximises likelihood vs the one that maximises log-likelihood?
3. Why **negative**, and why **average** (divide by the count)? Two reasons each, one sentence.
4. **Predict before running:** a model that knew nothing (every character equally likely, 1/27) would score an average NLL of ___. Compute it on paper.

In [ ]:
# predict: uniform model's average NLL = ___ ; our bigram model's will be ___ (higher or lower? roughly?)
log_likelihood = 0.0
n = 0
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1, ix2 = stoi[ch1], stoi[ch2]
        prob = P[ix1, ix2]
        log_likelihood += ___
        n += 1

nll = -log_likelihood
print(f'{log_likelihood=}')
print(f'{nll=}')
print(f'average nll = {nll / n}')

5. Write `name_nll(word, P)`: the average NLL of a single name. Score `zak`, `emma` and `andrejq`. Which bigram makes `andrejq` so expensive? Print the probability of each bigram in it.
6. Now rebuild P **without** the +1 (`Pzero = N.float(); Pzero /= ...`) and score `andrejq` again. **Predict** first. What happened, and why?

In [ ]:
def name_nll(word, P):
    ___

# predict: andrejq with +1 ≈ ___ ; without +1 = ___
for w in ['zak', 'emma', 'andrejq']:
    print(w, name_nll(w, P))

7. **CS × Phil (in the chat, 3 min).** Without smoothing, the model says `jq` has probability **exactly 0**, because it never saw one.
   - Is "never observed" a good reason to say "impossible"? What's the danger, for the model and for a scientist?
   - Adding 1 to every count before you've seen anything is a famous move in probability: **Laplace's rule of succession** (he used it on the question "will the sun rise tomorrow?"). What is the +1 *assuming* about the world before the data arrive? Is that assumption justified, or just useful?
   - Link to last night: you asked how the problem of induction applies to neural nets. Where does it show up here?

## Wrap (5 min)
For the activity log: what you **learned**, what **surprised** you, and the **question** it opened. One sentence each, your words.

Then commit: `git add -A && git commit -m "makemore 1: bigram counts, sampling, NLL" && git push`